In [ ]:
import os
import time
from pymmcore_plus import CMMCorePlus
from PyQt5.QtWidgets import QApplication, QMainWindow, QScrollArea, QTabWidget, QFileDialog, QRadioButton, QFrame, QSpinBox, QLineEdit, QCheckBox, QPushButton, QLabel, QHBoxLayout, QVBoxLayout, QComboBox, QWidget, QTableWidget, QTableWidgetItem, QMessageBox, QInputDialog, QGridLayout, QMessageBox
from PyQt5.QtGui import QPixmap, QImage
from PyQt5.QtCore import QTimer, pyqtSignal, pyqtSlot, Qt, QThread
from PyQt5 import QtGui, QtCore, QtWidgets
import serial
import numpy as np
import pandas as pd
import sys
from PIL import Image
import csv
import cv2
from skimage import exposure
from collections import defaultdict
from matplotlib.figure import Figure
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
import copy


Arduino_port = "COM4"
Arduino_baud_rate = 115200
Arduino_timeout = 0.1  # Max wait time for response


def initialize_arduino(port, baud_rate, timeout):
    """Initialize the serial connection with Arduino."""
    try:
        arduino = serial.Serial(port, baud_rate, timeout=timeout)
        time.sleep(2)  # Allow time for Arduino reset
        print(f"Connected to {port} at {baud_rate} baud.")
        return arduino
    except serial.SerialException:
        print(f"Error: Could not open serial port {port}. Check connection.")
        return None

def handshake(arduino):
    """Perform a handshake to ensure Arduino is ready."""
    if arduino:
        for _ in range(5):  # Retry up to 5 times
            arduino.write(b"HELLO\n")  # Send handshake message
            time.sleep(0.5)  # Wait for response
            response = arduino.readline().decode('utf-8').strip()
            if response == "READY":
                print("Handshake successful! Arduino is ready.")
                return True
        print("Handshake failed! No response from Arduino.")
        return False
    return False

# Initialize and connect
arduino = initialize_arduino(Arduino_port, Arduino_baud_rate, Arduino_timeout)

if arduino and handshake(arduino):
    print("Serial connection established. Ready for commands.")
else:
    print("Failed to establish a connection. Exiting.")
    if arduino:
        arduino.close()
    exit()


def send_to_arduino(data):
    """Send a command (either 'HELLO' for handshake or an integer)."""
    arduino.write(f"{data}\n".encode())  # Send data
    time.sleep(0.1)  # Short delay for response
    response = arduino.readline().decode().strip()  # Read response

    if response:
        print(f"Arduino Response: {response}")
    else:
        print("No response received.")

def set_voltage(set_value):
    setVoltage = int(set_value)
    print(setVoltage)
    return


                                                #Connect NUMATO Relay

Relay_port = 'COM6'
Relay_baudrate = 19200
Relay_timeout = 0.1

def show_device_status(connected):
    """Updates the device connection status."""
    if connected:
        print("Numato relay correctly connected")
        #self.device_status_label.setStyleSheet("color: green")
    else:
        print("Numato relay NOT correctly connected")
        #self.device_status_label.setStyleSheet("color: red")

def init_serial_port(relayport, relay_Baudrate, relay_timeout):
    """Initializes the serial port and returns the object."""
    try:
        # Try connecting to COM3 port (modify as per your port)
        obj = serial.Serial(relayport, relay_Baudrate, timeout=relay_timeout)
        obj.write(b'')  # Optional: send initial empty message
        obj.flush()
        obj.write_terminator = b'\r'  # Equivalent of 'CR' in MATLAB
        show_device_status(True)
        return obj
    except serial.SerialException as e:
        show_device_status(False)
        print('Error', 'Could not connect to the relay')
        sys.exit(1)

Numato_device = init_serial_port(Relay_port, Relay_baudrate, Relay_timeout)



class QToggleButton(QPushButton):
    def __init__(self, text='', parent=None):
        super().__init__(text, parent)
        
        # Ensure the button is checkable
        self.setCheckable(True)
        
        # Set the button to be checked by default
        self.setChecked(False)
        self.setStyleSheet("background-color: #bb283a;")



Microscope = {}
Microscope['mmc'] = CMMCorePlus.instance()
Microscope['mmc'].loadSystemConfiguration("C:\\MATLAB Microscope\\AmoghMMConfig_Hamamatsu.cfg")



class VideoThread(QThread):
    change_pixmap_signal = pyqtSignal(np.ndarray)

    def __init__(self):
        super().__init__()
        self._run_flag = True
        self._record_flag = False
        self.out = None  # cv2.VideoWriter instance
        self.video_directory = "C:/Users/Cell Culture Scope/Downloads/Videos"
        self.video_filename = self.get_unique_filename(self.video_directory)
        self.fourcc = cv2.VideoWriter_fourcc(*'MJPG')  # or 'MJPG', 'MP4V', etc.
        self.fps = 10  # adjust based on your capture rate
        self.frame_size = None

        # re-initialize all microscope attributes here for video-making
        self.mmc = Microscope['mmc']
        self.camera = self.mmc.getCameraDevice()
        self.DIAshutter = 'TIDiaShutter'
        self.focus = self.mmc.getFocusDevice()
        self.stage = self.mmc.getXYStageDevice()
        self.PFS = self.mmc.getAutoFocusDevice()
        self.EPIshutter = 'TIEpiShutter'
        #self.IntensiLightShutter = 'IntensiLightShutter'
        self.DIAlamp = 'TIDiaLamp'
        self.scope = 'TIScope'
        self.zoom = 'TINosePiece'
        self.filter = 'TIFilterBlock1'
        self.lightpath = 'TILightPath'
        self.PFS_offset = 'TIPFSOffset'
        self.core = 'Core'
        self.camerapath = '2-Left100'
        self.mmc.setProperty(self.camera, "CONVERSION FACTOR COEFF", "0.5")
        #self.mmc.setProperty(self.camera, 'Gain', 10)



    def run(self):
        # prepare microscope for video acquisition
        self.mmc.setProperty(self.DIAshutter,'State',0)
        self.mmc.setProperty(self.EPIshutter,'State',0)
        self.mmc.setProperty(self.core,'Shutter',self.DIAshutter)
        self.mmc.setProperty(self.lightpath,'Label',self.camerapath)
        self.mmc.initializeCircularBuffer()
        self.mmc.prepareSequenceAcquisition(self.camera)
        self.mmc.waitForDevice(self.DIAshutter)
        self.mmc.waitForDevice(self.camera)
        interval = 100 # image per X ms
        self.mmc.startContinuousSequenceAcquisition(interval)
        self.Sequencing = self.mmc.isSequenceRunning() # Outputs 1 to indicate video sequence is running
        while self._run_flag and self.Sequencing:
            if self.mmc.getRemainingImageCount() > 0:
                liveimage = self.mmc.getLastImage()  #Retrieves last image taken
                image_width = self.mmc.getImageWidth()
                image_height = self.mmc.getImageHeight()
                pixelType = np.uint16
                self.final_image = self.convert_raw_np(liveimage, image_width, image_height, pixelType)
                self.change_pixmap_signal.emit(self.final_image)

                if self._record_flag:
                    
                    if self.out is None:
                        self.video_filename = self.get_unique_filename(self.video_directory)
                        self.frame_size = (image_width, image_height)
                        self.out = cv2.VideoWriter(self.video_filename, 
                                                   self.fourcc, 
                                                   self.fps, 
                                                   self.frame_size, 
                                                   isColor=True
                                                  )
                    if self.out is not None and self.out.isOpened():
                        frame_rescaled = exposure.rescale_intensity(self.final_image, in_range='image', out_range='uint8').astype(np.uint8)
                        frame_to_save = cv2.cvtColor(frame_rescaled, cv2.COLOR_GRAY2BGR)
                        #frame_to_save = (self.final_image / self.final_image.max() * 255).astype(np.uint8)
                        try:
                            self.out.write(frame_to_save)
                        except Exception as e:
                            print(f"⚠️ Exception during video write: {e}")
                            self.handle_video_write_error()
                    elif self.out is not None and not self.out.isOpened():
                        print("⚠️ Failed to initialize VideoWriter")
                        self.out = None

        self.mmc.stopSequenceAcquisition(self.camera) #Stop acquisition and shut down capture system
        self.mmc.clearCircularBuffer()
        if self.out:
            self.out.release()
    

    def convert_raw_np(self, raw_img, img_width, img_height, pixel_Type):
        """Convert from an raw image to QPixmap"""
        # Assuming rawImage is a byte array
        rawImage = np.frombuffer(raw_img, dtype= pixel_Type)
        rawImage = rawImage.reshape((img_height, img_width)).T
        # Now assuming rawImage is a numpy array
        self.adjusted_image = exposure.rescale_intensity(rawImage)
        return self.adjusted_image
    
    def start_recording(self):
        self._record_flag = True
        

    def stop_recording(self):
        self._record_flag = False
        if self.out:
            self.out.release()
            self.out = None

    def get_unique_filename(self, directory, base="output", ext=".avi"):
        # Ensure the directory exists
        if not os.path.exists(directory):
            os.makedirs(directory)
        
        i = 1
        filename = os.path.join(directory, f"{base}{ext}")
        
        while os.path.exists(filename):
            filename = os.path.join(directory, f"{base}_{i}{ext}")
            i += 1
            
        return filename
    
    def handle_video_write_error(self):
        print("⚠️ Handling video write error: stopping recording")
        self._record_flag = False
        if self.out:
            self.out.release()
            self.out = None

    def stop(self):
        # Sets run flag to False and waits for thread to finish
        
        self._record_flag = False
        if self.out:
            self.out.release()
            self.out = None
        self._run_flag = False
        self.wait()



class DropletWorker(QThread):


    def __init__(self, operation, input_1, input_2 = None, 
                 purge_duration = 0, flow_duration = 0, drive_duration = 0, chemostat_number = 4, 
                 PWM_duration1 = 0.05, PWM_duration2 = 0.05, PWM_totalduration = 5):
        super().__init__()
        #self._run_flag = True
        self.Numato_port = Numato_device
        self.operation = operation  # "purge", "generate", "drive", "characterize"
        self.input_1 = input_1
        self.input_2 = input_2
        self.chemostat = chemostat_number
        self.Purge_duration = purge_duration
        self.Flow_duration = flow_duration
        self.Drive_duration = drive_duration
        self.PWM_duration1 = PWM_duration1
        self.PWM_duration2 = PWM_duration2
        self.PWM_totalduration = PWM_totalduration
    
    def run(self):
        if self.operation == "purge":
            self.purge_inlet(self.input_1, self.input_2)
        elif self.operation == "generate":
            self.generate_droplet(self.input_1, self.input_2)
        elif self.operation == "drive":
            self.drive_droplet(self.chemostat)
        elif self.operation == "characterize":
            self.characterize_droplet(self.chemostat)
        elif self.operation == "wash":
            self.wash_step(self.input_1, self.input_2)
        elif self.operation == "PWM":
            self.PWM(self.input_1, self.input_2, self.PWM_duration1, self.PWM_duration2, self.PWM_totalduration)
        else:
            print("Invalid operation name provided")
    

    def characterize_droplet(self, chemostat_number):
        self.control_valve(8, state = False)
        self.control_valve(9, state = False)
        self.control_valve(10, state = False)
        self.control_valve(11, state = False)
        print(f"purging worker thread {self.Purge_duration}")
        time.sleep(3)
        self.control_valve(15, state = False)
        self.control_valve(7, state = False)
        time.sleep(self.Purge_duration)

        self.control_valve(2, state = True)
        self.control_valve(5, state = True)
        time.sleep(0.5)
        self.control_valve(7, state = True)
        self.control_valve(4, state = False)
        self.control_valve(3, state = False)
        time.sleep(self.Flow_duration)
        self.control_valve(3, state = True)
        self.control_valve(4, state = True)
        time.sleep(2)
        self.control_valve(15, state = True)
        #self.control_valve(input_2, state = True)

        self.control_valve(2, state = False)
        self.control_valve(5, state = False)
        self.control_valve(chemostat_number + 7, state = True)

        send_to_arduino(2)
        print(f"driving worker thread {self.Drive_duration}")
        time.sleep(self.Drive_duration)
        self.control_valve(chemostat_number + 7, state = False)

        send_to_arduino(3)
        self.control_valve(4, state = False)
        time.sleep(1)
        self.control_valve(4, state = True)


    def purge_inlet(self, input_1, input_2):

        self.control_valve(input_1 + 11, state = False)
        time.sleep(1)
        self.control_valve(7, state = False)
        self.control_valve(input_2 + 11, state = False)
        print(f'purging {self.Purge_duration}')
        time.sleep(self.Purge_duration)
        self.control_valve(input_1 + 11, state = True)
        self.control_valve(input_2 + 11, state = True)
        self.control_valve(7, state = True)
        time.sleep(20)

    
    def generate_droplet(self, input_1, input_2):

        self.control_valve(2, state = True)
        self.control_valve(5, state = True)

        self.control_valve(15, state = False)
        time.sleep(0.1)
        
        self.control_valve(3, state = False)
        self.control_valve(4, state = False)
        print(f'generating {self.Flow_duration}')
        time.sleep(self.Flow_duration)
        self.control_valve(3, state = True)
        self.control_valve(4, state = True)
        self.control_valve(15, state = True)



    def drive_droplet(self, Chemostat_number):

        self.control_valve(2, state = False)
        self.control_valve(5, state = False)
        self.control_valve(Chemostat_number + 7, state = True)
        send_to_arduino(2)
        print(f'driving {self.Drive_duration}s to {self.chemostat}')
        time.sleep(self.Drive_duration)
        send_to_arduino(3)
        self.control_valve(Chemostat_number + 7, state = False)
        self.control_valve(4, state = False)
        time.sleep(3)
        self.control_valve(4, state = True)

    def wash_step(self, input_1, input_2):
        self.control_valve(input_1 + 11, state = False)
        self.control_valve(input_2 + 11, state = False)
        time.sleep(4)
        self.control_valve(input_2 + 11, state = True)
        self.control_valve(7, state = False)
        time.sleep(5)
        self.control_valve(input_1 + 11, state = True)
        self.control_valve(7, state = True)

    def PWM(self, input_1 = 12, input_2 = 13, duration_1 = 0.05, duration_2 = 0.05, total_duration = 5.0):
        
        #Clear out channel with buffer
        self.control_valve(6, state = False)
        self.control_valve(7, state = False)
        self.control_valve(15, state = False)
        time.sleep(5)
        self.control_valve(15, state = True)
        self.control_valve(6, state = True)

        #Start PWM
        start_time = time.time()
        while time.time() - start_time < total_duration:
            self.control_valve(input_1, state = False)
            time.sleep(duration_1)
            self.control_valve(input_1, state = True)
            time.sleep(0.1)
            self.control_valve(input_2, state = False)
            time.sleep(duration_2)
            self.control_valve(input_2, state = True)
            time.sleep(0.1)

        self.control_valve(7, state = True)

        



    def send_relay_command(self, command):
        """Sends a command to the serial device."""
        if self.Numato_port and self.Numato_port.is_open:
            try:
                self.Numato_port.write(f"{command}\r".encode('utf-8'))
                #self.Numato_port.flush()
                time.sleep(0.005)
            except serial.SerialException as e:
                print(self, 'Error', 'Worker thread failed to communicate with the device')
    
    def control_valve(self, idx, state):
        """Handles Control button toggle actions."""
        # Relay index mapping for serial communication
        relay_id = self.get_relay_id(idx)

        if state:
            # Turn the valve on
            self.send_relay_command(f"relay off {relay_id}")
        else:
            # Turn the valve off
            self.send_relay_command(f"relay on {relay_id}")

    def get_relay_id(self, idx):
        """Maps relay 10-21 to A-L for serial communication, keeps others unchanged."""
        if idx <= 9:  # For relays 0 to 9
            return str(idx)
        else:  # For relays 10 to 21, map to A-L
            return chr(ord('A') + (idx - 10))









                                                                        #MAIN MICROSCOPE APP












class MicroscopeControlGUI(QMainWindow):
    def __init__(self):
        super().__init__()
        self.initUI()
        self.resize(600, 600)
        self.Numato_port = Numato_device

    def initUI(self):
        # Change working directory
        os.chdir('C:/Users/Cell Culture Scope/Documents/MATLAB')

        # Set dark mode palette
        dark_palette = QtGui.QPalette()
        dark_palette.setColor(QtGui.QPalette.Window, QtGui.QColor(53, 53, 53))
        dark_palette.setColor(QtGui.QPalette.WindowText, QtCore.Qt.white)
        dark_palette.setColor(QtGui.QPalette.Base, QtGui.QColor(25, 25, 25))
        dark_palette.setColor(QtGui.QPalette.AlternateBase, QtGui.QColor(53, 53, 53))
        dark_palette.setColor(QtGui.QPalette.ToolTipBase, QtCore.Qt.white)
        dark_palette.setColor(QtGui.QPalette.ToolTipText, QtCore.Qt.white)
        dark_palette.setColor(QtGui.QPalette.Text, QtCore.Qt.white)
        dark_palette.setColor(QtGui.QPalette.Button, QtGui.QColor(53, 53, 53))
        dark_palette.setColor(QtGui.QPalette.ButtonText, QtCore.Qt.white)
        dark_palette.setColor(QtGui.QPalette.BrightText, QtCore.Qt.red)
        dark_palette.setColor(QtGui.QPalette.Link, QtGui.QColor(42, 130, 218))
        dark_palette.setColor(QtGui.QPalette.Highlight, QtGui.QColor(42, 130, 218))
        dark_palette.setColor(QtGui.QPalette.HighlightedText, QtCore.Qt.black)
        
        self.setPalette(dark_palette)

        # Create a tab widget and add it to the main window
        self.tabs = QTabWidget()
        self.setCentralWidget(self.tabs)

        # Add the first tab (Microscope control) and other tabs as needed
        self.microscope_tab = QWidget()
        self.tabs.addTab(self.microscope_tab, "Microscope Control")

        

        #central_widget = QWidget()
        #self.setCentralWidget(central_widget)
        self.microscope_tab_layout = QGridLayout()
        self.microscope_tab.setLayout(self.microscope_tab_layout)

        # Slider Initialization
        """self.dialampslider = QSlider(self)
        self.dialampslider.setMinimum(1)
        self.dialampslider.setMaximum(24)
        self.dialampslider.setValue(2)
        steps_dia = [1/50, 24/50]  # Step values
        self.dialampslider.setSingleStep(int(steps_dia[0]*100))"""

        """self.EMslider = QSlider(self)
        self.EMslider.setMinimum(25)
        self.EMslider.setMaximum(51)
        self.EMslider.setSingleStep(1)"""

        # Micro-Manager Initialization
        self.mmc = Microscope['mmc']
        #self.mmc.loadSystemConfiguration("C:\\MATLAB Microscope\\AmoghMMConfig.cfg")

        Devices = self.mmc.getLoadedDevices()
        Devices_list = [Devices[i] for i in range(len(Devices))]

        self.camera = self.mmc.getCameraDevice()
        self.DIAshutter = self.mmc.getShutterDevice()
        self.focus = self.mmc.getFocusDevice()
        self.stage = self.mmc.getXYStageDevice()
        self.PFS = self.mmc.getAutoFocusDevice()
        self.EPIshutter = 'TIEpiShutter'
        self.DIAlamp = 'TIDiaLamp'
        self.scope = 'TIScope'
        self.zoom = 'TINosePiece'
        self.filter = 'TIFilterBlock1'
        self.lightpath = 'TILightPath'
        self.PFS_offset = 'TIPFSOffset'
        self.core = 'Core'
        #self.mmc.setProperty('IntensiLightShutter', 'State', 1)

        CamProperties = self.mmc.getDevicePropertyNames(self.camera)
        self.CamProperties_list = [CamProperties[i] for i in range(len(CamProperties))]

        ScopeProperties = self.mmc.getDevicePropertyNames(self.scope)
        self.ScopeProperties_list = [ScopeProperties[i] for i in range(len(ScopeProperties))]

        DIALampProperties = self.mmc.getDevicePropertyNames(self.DIAlamp)
        self.DIALampProperties_list = [DIALampProperties[i] for i in range(len(DIALampProperties))]

        ZoomProperties = self.mmc.getDevicePropertyNames(self.zoom)
        self.ZoomProperties_list = [ZoomProperties[i] for i in range(len(ZoomProperties))]
        

        self.zoom4x = '1-(Achromat) 4x NA 0.10 Dry'
        self.zoom10x = '2-(Achromat) 10x NA 0.25 Dry'
        self.zoom20x = '3-(Achromat) 20x NA 0.40 Dry'
        self.zoom40x = '4-S Plan Fluor 40x NA 0.60 Dry'
        self.zoom60x = '5-Plan Apo 60x NA 1.40 Oil'
        self.zoomempty = '6-Unknown'
        

        FilterProperties = self.mmc.getDevicePropertyNames(self.filter)
        self.FilterProperties_list = [FilterProperties[i] for i in range(len(FilterProperties))]
        print(self.FilterProperties_list)


        CoreProperties = self.mmc.getDevicePropertyNames(self.core)
        self.CoreProperties_list = [CoreProperties[i] for i in range(len(CoreProperties))]

        DIAShutterProperties = self.mmc.getDevicePropertyNames(self.DIAshutter)
        self.DIAShutterProperties_list = [DIAShutterProperties[i] for i in range(len(DIAShutterProperties))]

        EPIShutterProperties = self.mmc.getDevicePropertyNames(self.EPIshutter)
        self.EPIShutterProperties_list = [EPIShutterProperties[i] for i in range(len(EPIShutterProperties))]

        StageProperties = self.mmc.getDevicePropertyNames(self.stage)
        self.StageProperties_list = [StageProperties[i] for i in range(len(StageProperties))]

        LightPathProperties = self.mmc.getDevicePropertyNames(self.lightpath)
        self.LightPathProperties_list = [LightPathProperties[i] for i in range(len(LightPathProperties))]

        self.eyepath = '1-Eye100'
        self.camerapath = '2-Left100'

        # Initialization for the stage controls
        """self.xyfastlight.setChecked(True)
        self.xymediumlight.setChecked(False)
        self.xyslowlight.setChecked(False)
        self.zfastlight.setChecked(True)
        self.zmediumlight.setChecked(False)
        self.zslowlight.setChecked(False)"""

        # Camera Initialization
        self.mmc.setProperty(self.camera, 'Sensor Cooler', 'ON')
        self.mmc.setProperty(self.camera, 'Exposure', 20)
        self.mmc.setProperty(self.camera, 'MINIMUM ACQUISITION TIMEOUT', 500)
        self.mmc.setProperty(self.camera, 'Binning', '4x4')
        self.mmc.setProperty(self.lightpath, 'Label', self.eyepath)
        self.mmc.setProperty(self.camera, 'ScanMode', 1)
        allowed_values = self.mmc.getAllowedPropertyValues(self.camera, "CONVERSION FACTOR COEFF")
        print(allowed_values)

        self.eyepathlight = QPushButton("Path to Eye")
        self.eyepathlight.setChecked(True)        
        self.eyepathlight.clicked.connect(self.PathtoCamera)
        
        self.camerapathlight = QPushButton("Path to Camera")
        self.camerapathlight.setChecked(True)
        self.camerapathlight.clicked.connect(self.PathtoCamera)

        

        self.mmc.setProperty(self.camera, 'CONVERSION FACTOR COEFF', 0.5)
        #self.gaintext = QLineEdit(self)
        #self.gaintext.setText('10')


        self.mmc.setProperty(self.camera, 'PixelType', '16bit')

        # Zoom Initialization
        self.Zoom_list = QComboBox()
        self.Zoom_list.addItems([self.zoom4x, self.zoom10x, self.zoom20x, self.zoom40x, self.zoom60x, self.zoomempty])
        self.Zoom_list.currentTextChanged.connect(self.Set_zoom)
        #self.microscope_tab_layout.addWidget(self.Zoom_list)
        zoom_ini = self.mmc.getProperty(self.zoom, 'Label')
        #print(zoom_ini)
        """if zoom_ini == self.zoom4x:
            self.zoomlight4x.setChecked(True)
        elif zoom_ini == self.zoom10x:
            self.zoomlight10x.setChecked(True)
        elif zoom_ini == self.zoom20x:
            self.zoomlight20x.setChecked(True)
        elif zoom_ini == self.zoom40x:
            self.zoomlight40x.setChecked(True)
        elif zoom_ini == self.zoom60x:
            self.zoomlight60x.setChecked(True)"""

        # Light Path Initialization
        self.mmc.setProperty(self.lightpath, 'Label', self.eyepath)
        self.eyepathlight.setChecked(True)
        self.camerapathlight.setChecked(False)

        

        # DIA Lamp Initialization
        self.mmc.setProperty(self.DIAlamp, 'ComputerControl', 'On')
        """self.dialampmanuallight = QPushButton(self)
        self.dialampmanuallight.setChecked(False)
        self.dialampsoftwarelight = QPushButton(self)
        self.dialampsoftwarelight.setChecked(True)"""

        self.mmc.setProperty(self.DIAlamp, 'Intensity', 4)
        self.mmc.setProperty(self.DIAlamp, 'State', 0)
        self.dialamponlight = QToggleButton()
        self.dialamponlight.setText("DIA Lamp")
        self.dialamponlight.clicked.connect(self.DIAlamp_ON)
        """self.dialampvalue = QLineEdit(self)
        self.dialampvalue.setText('2')"""

        # Shutter Initialization
        self.mmc.setProperty(self.DIAshutter, 'State', 0)
        self.mmc.setProperty(self.EPIshutter, 'State', 0)

        #DIA Shutter
        self.DIAshutterbutton = QToggleButton("DIA Shutter")
        self.DIAshutterbutton.clicked.connect(self.toggle_DIA_shutter)

        #EPI Shutter
        self.EPIshutterbutton = QToggleButton("EPI Shutter")
        self.EPIshutterbutton.clicked.connect(self.toggle_EPI_shutter)



            #Adding all shutter buttons
        self.light_layout = QGridLayout()
        self.light_layout.addWidget(self.dialamponlight, 0, 0)
        self.light_layout.addWidget(self.DIAshutterbutton, 0, 1)
        self.light_layout.addWidget(self.EPIshutterbutton, 0, 2)
        self.light_layout.addWidget(self.Zoom_list, 1, 0)

                                                            # Change Filters
        
        filter_buttons_layout = QtWidgets.QHBoxLayout()

        # Filter Initialization
        filter_ini = self.mmc.getProperty(self.filter, 'Label')
        """if filter_ini == self.filter_Cy5_1:
            self.cy5filterlight.setChecked(True)
        elif filter_ini == self.filter_Cy3_2:
            self.cy3filterlight.setChecked(True)
        elif filter_ini == self.filter_DIA_6:
            self.DIAfilterlight.setChecked(True)"""
        self.filterNames = {
            1: '1- FITC',
            2: '2- DAPI',
            3: '3- BFP-A',
            4: '4- Cy5',
            5: '5- Cy3',
            6: '6- DIA'
        }

        filter_buttons = {}

        for filter_key, filter_value in self.filterNames.items():
            button = QPushButton(filter_value)
            button.clicked.connect(lambda checked, fn=filter_key: self.change_filter(fn))
            filter_buttons[filter_key] = button
        
        row = 2
        col = 0
        for button in filter_buttons.values():
            self.light_layout.addWidget(button, row, col)
            col += 1
            if col > 2:
                col = 0
                row += 1

        
        


                                                            # Save and display positions

            # Reading stage position
        self.xcoord = self.mmc.getXPosition(self.stage)
        self.ycoord = self.mmc.getYPosition(self.stage)
        self.zcoord = self.mmc.getPosition()

            # Setting up positions table

        self.positions = []
        self.positions_layout = QVBoxLayout()
        self.positions_buttons_list = QHBoxLayout()

        
        self.positions_max = QLabel('Saved Positions (Max 8):')
        self.positions_layout.addWidget(self.positions_max)

        # Table to display positions
        self.Positions_table = QTableWidget(self)
        self.Positions_table.setRowCount(8)  # Maximum 8 positions
        self.Positions_table.setColumnCount(3)  # X, Y, Z
        self.Positions_table.setHorizontalHeaderLabels(['X', 'Y', 'Z'])
        self.positions_layout.addWidget(self.Positions_table)

        self.save_Position_button = QPushButton("Add Position", self)
        self.save_Position_button.clicked.connect(self.save_Position)
        self.positions_buttons_list.addWidget(self.save_Position_button)

        self.replacePositionButton = QPushButton('Replace Specific Position', self)
        self.replacePositionButton.clicked.connect(self.replacePosition)
        self.positions_buttons_list.addWidget(self.replacePositionButton)
        
        self.clearButton = QPushButton('Clear All Positions', self)
        self.clearButton.clicked.connect(self.clearPositions)
        self.positions_buttons_list.addWidget(self.clearButton)

        self.positions_layout.addLayout(self.positions_buttons_list)



                                                            # Stage Slots


         # Setting stage movespeed for control via app
        self.stagespeedbutton = QComboBox()
        self.stagefast   =   1000  # 1000um per click
        self.stagemedium =   100   # 100um per click
        self.stageslow   =   10    # 10um per click
        self.stagespeedbutton.addItems([str(self.stageslow), str(self.stagemedium), str(self.stagefast)])

        self.stage_speed = self.stagefast
        self.stagespeedbutton.currentTextChanged.connect(self.Set_stage_speed)
        #print(self.xcoord, self.ycoord, self.zcoord)

            # Stage movement buttons on app

        self.Xplus = QPushButton("X+")
        self.Xplus.clicked.connect(lambda: self.mmc.setXYPosition((self.xcoord + self.stage_speed), self.ycoord))

        self.Xminus = QPushButton("X-")
        self.Xminus.clicked.connect(lambda: self.mmc.setXYPosition((self.xcoord - self.stage_speed), self.ycoord))

        self.Yplus = QPushButton("Y+")
        self.Yplus.clicked.connect(lambda: self.mmc.setXYPosition(self.xcoord, (self.ycoord + self.stage_speed)))

        self.Yminus = QPushButton("Y-")
        self.Yminus.clicked.connect(lambda: self.mmc.setXYPosition(self.xcoord, (self.ycoord - self.stage_speed)))

        self.Zplus = QPushButton("Z+")
        self.Zplus.clicked.connect(lambda: self.mmc.setPosition((self.zcoord + self.stage_speed)))

        self.Zminus = QPushButton("Z-")
        self.Zminus.clicked.connect(lambda: self.mmc.setPosition((self.zcoord - self.stage_speed)))

        stage_layout = QGridLayout()
        stage_layout.addWidget(self.Xplus, 0, 0)
        stage_layout.addWidget(self.Xminus, 0, 1)
        stage_layout.addWidget(self.Yplus, 1, 0)
        stage_layout.addWidget(self.Yminus, 1, 1)
        stage_layout.addWidget(self.Zplus, 2, 0)
        stage_layout.addWidget(self.Zminus, 2, 1)
        stage_layout.addWidget(self.stagespeedbutton, 3, 0)

        


                                                            # Pipeline to display/save images
        
        self.snap_Button = QPushButton("Snap Image")
        self.snap_Button.clicked.connect(self.snap_DIA_image)
        self.image_Live = QLabel('No image loaded', self)
        
        self.live_Button = QToggleButton("Live Image")
        self.live_Button.clicked.connect(self.startstoplive_imaging)

        self.record_button = QPushButton("Start Recording")
        self.record_button.setCheckable(True)
        self.record_button.clicked.connect(self.handle_record_button)

        # Create a button to load the image
        self.save_Button = QPushButton('Save Image', self)
        self.save_Button.clicked.connect(self.save_image)


        #Adding all stage widgets and image buttons to layout
        self.video_button_layout = QHBoxLayout()
        self.video_button_layout.addWidget(self.snap_Button)
        self.video_button_layout.addWidget(self.live_Button)
        self.video_button_layout.addWidget(self.save_Button)
        self.video_button_layout.addWidget(self.record_button)


                                                            

                                                            # Valve Controls
        

        # Layouts
        self.valves_layout = QVBoxLayout()

        # Grid self.valves_layout for Control buttons
        self.valve_buttons_layout = QGridLayout()

        # Labels for relays 1 to 21 (numeric)
        relay_labels = [str(i+1) for i in range(21)]  # '1' to '21'

        # Create control buttons 1 to 21
        self.controls = []
        for i in range(21):
            control = QPushButton(f"Control {relay_labels[i]}")
            control.setCheckable(True)
            control.setStyleSheet("background-color: red")
            control.clicked.connect(lambda state, idx=i: self.control_valve(idx, state))
            self.controls.append(control)
            row = i // 3
            col = i % 3
            self.valve_buttons_layout.addWidget(control, row, col)

        self.valves_layout.addLayout(self.valve_buttons_layout)

        # "Stop All" button
        self.stop_all_button = QPushButton("Stop All")
        self.stop_all_button.setStyleSheet("background-color: black")
        self.stop_all_button.clicked.connect(self.stop_all_callback)
        self.valves_layout.addWidget(self.stop_all_button)

        # "All On" button
        self.all_on_button = QPushButton("All On")
        self.all_on_button.setStyleSheet("background-color: blue")
        self.all_on_button.clicked.connect(self.all_on_callback)  # New button
        self.valves_layout.addWidget(self.all_on_button)

        # Droplet Characterization 
        self.purge_duration_label = QLabel("Purge Duration:")
        self.purge_duration_Input = QLineEdit()
        self.purge_duration_Input.setText("0") #Default


        self.flow_duration_label = QLabel("Aqueous Flow Duration:")
        self.flow_duration_Input = QLineEdit()
        self.flow_duration_Input.setText("0") #Default
      

        self.drive_duration_label = QLabel("Drive droplet Duration:")
        self.drive_duration_Input = QLineEdit()
        self.drive_duration_Input.setText("0") #Default

        self.inlet_label = QLabel("Chemostat_number")
        self.inlet_Input = QLineEdit()
        self.inlet_Input.setText("1") #Default
        
        self.PWM_duration1_label = QLabel("PWM Inlet 1 Duration:")
        self.PWM_duration1_Input = QLineEdit()
        self.PWM_duration1_Input.setText("0.05") #Default

        self.PWM_duration2_label = QLabel("PWM Inlet 2 Duration:")
        self.PWM_duration2_Input = QLineEdit()
        self.PWM_duration2_Input.setText("0.05") #Default

        self.PWM_totalduration_label = QLabel("PWM runtime:")
        self.PWM_totalduration_Input = QLineEdit()
        self.PWM_totalduration_Input.setText("5") #Default

        self.PWM_button = QPushButton("Run PWM")
        self.PWM_button.setStyleSheet("background-color: yellow")
        self.PWM_button.clicked.connect(lambda: self.PWM_droplet())  # New button


        self.master_droplet_input = 2

        self.Generate_drop_button = QPushButton("Generate droplet")
        self.Generate_drop_button.setStyleSheet("background-color: blue")
        self.Generate_drop_button.clicked.connect(lambda: self.Characterize_Droplet(input = 3))  # New button
        self.valves_layout.addWidget(self.purge_duration_label)
        self.valves_layout.addWidget(self.purge_duration_Input)
        self.valves_layout.addWidget(self.flow_duration_label)
        self.valves_layout.addWidget(self.flow_duration_Input)
        self.valves_layout.addWidget(self.drive_duration_label)
        self.valves_layout.addWidget(self.drive_duration_Input)
        self.valves_layout.addWidget(self.inlet_label)
        self.valves_layout.addWidget(self.inlet_Input)
        self.valves_layout.addWidget(self.Generate_drop_button)

        self.valves_layout.addWidget(self.PWM_duration1_label)
        self.valves_layout.addWidget(self.PWM_duration1_Input)
        self.valves_layout.addWidget(self.PWM_duration2_label)
        self.valves_layout.addWidget(self.PWM_duration2_Input)
        self.valves_layout.addWidget(self.PWM_totalduration_label)
        self.valves_layout.addWidget(self.PWM_totalduration_Input)
        self.valves_layout.addWidget(self.PWM_button)

        
        


        # Add all layouts to main layout
        self.microscope_tab_layout.addLayout(self.light_layout, 0, 0)
        self.image_path_layout = QHBoxLayout()
        self.image_path_layout.addWidget(self.camerapathlight)
        self.image_path_layout.addWidget(self.eyepathlight)
        self.microscope_tab_layout.addLayout(self.image_path_layout, 4, 0)

        self.microscope_tab_layout.addLayout(self.video_button_layout, 5, 0)

        self.microscope_tab_layout.addWidget(self.image_Live, 6, 0)

        self.microscope_tab_layout.addLayout(stage_layout, 7, 0)

        self.microscope_tab_layout.addLayout(self.positions_layout, 0, 1)

        self.microscope_tab_layout.addLayout(self.valves_layout, 1, 1, -1, 1)
        

        self.total_positions = 8

        '''for i in range(self.total_positions):
            self.position_index[i] = QLineEdit()

            self.microscope_tab_layout.addWidget(self.position_index[i], row + 7 + i, 0)'''
        


        self.Timelapse_Setup_tab = QWidget()
        self.tabs.addTab(self.Timelapse_Setup_tab, "Timelapse Setup")
        self.Timelapse_layout = QVBoxLayout()
        self.Timelapse_Setup_tab.setLayout(self.Timelapse_layout)
        
        
        self.Exposures_layout = QHBoxLayout()
        self.Filters_list = QVBoxLayout()
        Filter_List_label = QLabel("Available filters:")
        Filter_List_label.setStyleSheet("font-size: 16px; font-weight: bold; color: white;")
        self.Filters_list.addWidget(Filter_List_label)
        for filter_key, filter_value in self.filterNames.items():
            self.Filters_list.addWidget(QLabel(filter_value))
        self.Exposures_table = QTableWidget(len(self.filterNames), 2)
        self.Exposures_table.setHorizontalHeaderLabels(["Select Filter (1-6)", "Exposure (1-1000 ms)"])

        for row in range(len(self.filterNames)):
            # First column: values between 1 and 6
            spinbox1 = QSpinBox()
            spinbox1.setRange(1, 6)
            spinbox1.setValue(1)  # Default value
            self.Exposures_table.setCellWidget(row, 0, spinbox1)

            # Second column: values between 1 and 1000
            spinbox2 = QSpinBox()
            spinbox2.setRange(0, 1000)
            spinbox2.setValue(0)  # Default value
            self.Exposures_table.setCellWidget(row, 1, spinbox2)

        self.ExposuresTableLayout = QVBoxLayout()
        self.ExposuresTableLayout.addWidget(self.Exposures_table)
        

        self.Save_Exposures_button = QPushButton("Save Values")
        self.Save_Exposures_button.clicked.connect(self.read_Exposure_values)

        self.quick_EPI_filter = QSpinBox()
        self.quick_EPI_filter.setRange(1, 6)
        self.quick_EPI_filter.setValue(4)  # Default value
        self.quick_EPI_filter_value = self.quick_EPI_filter.value()

        self.quick_EPI_exposure = QLineEdit()
        self.quick_EPI_exposure.setText("100") #Default
        self.quick_EPI_exposure_value = int(self.quick_EPI_exposure.text())

        self.snap_EPI_button = QPushButton("Snap Fluorescent image")
        self.snap_EPI_button.clicked.connect(lambda: self.snap_EPI_image(Filter = self.quick_EPI_filter.value(), 
                                                                         Exposure_value = int(self.quick_EPI_exposure.text())
                                                                         )
                                                                         )

        self.Exposures_table.resizeColumnsToContents()
        self.Exposures_layout.setStretch(1, 1)  # First VBox
        self.Exposures_layout.setStretch(2, 1)  # Second VBox
        self.AdjustTableSize(self.Exposures_table)
        self.Exposures_layout.addLayout(self.Filters_list)
        self.Exposures_layout.addLayout(self.ExposuresTableLayout)
        self.Exposures_layout.addWidget(self.Save_Exposures_button)
        self.Exposures_layout.addWidget(self.snap_EPI_button)

        self.Exposures_layout.addWidget(QLabel("Select filter for quick image:"))
        self.Exposures_layout.addWidget(self.quick_EPI_filter)
        self.Exposures_layout.addWidget(QLabel("Exposure for quick image:"))
        self.Exposures_layout.addWidget(self.quick_EPI_exposure)
        self.Timelapse_layout.addLayout(self.Exposures_layout)
        


        # Inputs and Chemostats Loading Section
        self.Valve_protocol_layout = QHBoxLayout()     
        self.Valve_protocol_layout_1 = QVBoxLayout()
        self.Chemostat_inputs = []
        self.Chemostats = []
        self.step_sequences = []             # List of tables (each is a list of steps)
        self.experiment_loop_counts = []     # Number of loops for each table

        for i in range(4):  # Inputs 1 to 4
            textbox = QLabel(f"Input {i + 1}")
            spinbox = QSpinBox()
            spinbox.setRange(0, 5)
            spinbox.setValue(0)
            self.Valve_protocol_layout_1.addWidget(textbox)
            self.Valve_protocol_layout_1.addWidget(spinbox)
            self.Chemostat_inputs.append(spinbox)
        
        self.Valve_protocol_layout_2 = QVBoxLayout()
        
        for i in range(8):  # Chemostats 1 to 8
            checkbox = QCheckBox(f"Chemostat {i + 1}")
            self.Valve_protocol_layout_2.addWidget(checkbox)
            self.Chemostats.append(checkbox)
        
        self.Valve_protocol_layout_3 = QVBoxLayout()

        # Buttons
        self.add_Step_button = QPushButton("Add Step")
        self.clear_Steps_button = QPushButton("Clear All Steps")
        self.clear_last_button = QPushButton("Clear Last Step")
        self.export_button = QPushButton("Export to CSV")
        self.Valve_protocol_layout_3.addWidget(self.add_Step_button)
        self.Valve_protocol_layout_3.addWidget(self.clear_last_button)
        self.Valve_protocol_layout_3.addWidget(self.clear_Steps_button)
        self.Valve_protocol_layout_3.addWidget(self.export_button)
        
        
        
        # Table to Display Sequence
        self.Valve_protocol_layout_4 = QVBoxLayout()
        self.Chemostat_protocol_table = QTableWidget(self)  # 10 rows: 2 for inputs, 8 for rings
        self.Chemostat_protocol_table.setRowCount(10)
        self.Chemostat_protocol_table.setColumnCount(8)
        self.Chemostat_protocol_table.setVerticalHeaderLabels(
            ["Input 1", "Input 2"] + [f"Chemostat {i + 1}" for i in range(8)]
        )
        for col_index in range(8):
            self.Chemostat_protocol_table.setHorizontalHeaderItem(col_index, QTableWidgetItem(f"Step {col_index + 1}"))
            
            # Set placeholder content (empty cells)
        for col in range(8):
            for row in range(10):
                # Add placeholders to simulate empty cells
                if row < 2:  # First 2 rows for "Input 1" and "Input 2"
                    self.Chemostat_protocol_table.setItem(row, col, QTableWidgetItem(""))
                else:  # Remaining rows for rings
                    checkbox = self.create_centered_checkbox()
                    self.Chemostat_protocol_table.setCellWidget(row, col, checkbox)


        self.Chemostat_protocol_table.resizeColumnsToContents()
        
        self.Valve_protocol_layout_4.addWidget(self.Chemostat_protocol_table)
        
        # Connect Signals
        self.add_Step_button.clicked.connect(self.add_loading_step)
        self.clear_Steps_button.clicked.connect(self.clear_all_steps)
        self.clear_last_button.clicked.connect(self.clear_last_step)
        self.export_button.clicked.connect(self.export_to_csv)
        
        # Data storage
        self.current_protocol_table_step = 0
        self.Chemostat_protocol_steps = []


        self.Valve_protocol_layout.addLayout(self.Valve_protocol_layout_1)
        self.Valve_protocol_layout.addLayout(self.Valve_protocol_layout_2)
        self.Valve_protocol_layout.addLayout(self.Valve_protocol_layout_3)
        self.Valve_protocol_layout.addLayout(self.Valve_protocol_layout_4)
        self.Valve_protocol_layout.setStretch(0, 1)  # First VBox
        self.Valve_protocol_layout.setStretch(1, 1)  # Second VBox
        self.Timelapse_layout.addLayout(self.Valve_protocol_layout)


        # Long-term experiment layout

        # Set cycle interval
        self.interval_label = QLabel("Cycle interval in minutes:")
        self.cycle_Interval_Input = QLineEdit()
        self.cycle_Interval_Input.setText("0") #Default
        self.interval = int(self.cycle_Interval_Input.text())

        # Set number of cycles
        self.cycle_label = QLabel("No. of cycles:")
        self.cycle_Input = QLineEdit()
        self.cycle_Input.setText("0") #Default
        self.cycles = int(self.cycle_Input.text())

        self.start_experiment_button = QPushButton("Start Experiment")
        self.start_experiment_button.setStyleSheet("background-color: #2ac555;")
        self.start_experiment_button.clicked.connect(lambda: self.TimeLapse_Experiment(num_loops = self.cycles,
                                                                                       time_interval = self.interval,
                                                                                       positions_table = self.positions,
                                                                                       selected_exposures = self.selected_exposures,
                                                                                       chemostat_protocol_table = self.Chemostat_protocol_steps))
        self.stop_experiment_button = QPushButton("Stop Experiment")
        self.stop_experiment_button.setStyleSheet("background-color: #bb283a;")
        
        



        self.directory_input = QLineEdit(self)
        self.browse_button = QPushButton("Browse", self)
        self.browse_button.clicked.connect(self.browse_folder)

        self.TurnOnVolts = QPushButton('Voltage Signal')
        self.TurnOnVolts.setStyleSheet("background-color: #bb283a;")
        self.TurnOnVolts.setCheckable(True)
        self.TurnOnVolts.setChecked(False)
        self.TurnOnVolts.clicked.connect(self.alter_Arduino_state)

        self.voltagevalues = QComboBox()
        self.voltagevalues.addItems(['200', '400', '600', '800', '1000'])
        self.voltagevalues.currentTextChanged.connect(set_voltage)
        voltagelabel = QLabel('Operating voltage in V')
        

        self.experiment_layout = QGridLayout()
        self.experiment_layout.addWidget(self.interval_label, 0, 0)
        self.experiment_layout.addWidget(self.cycle_Interval_Input, 0, 2)
        self.experiment_layout.addWidget(self.cycle_label, 0, 3)
        self.experiment_layout.addWidget(self.cycle_Input, 0, 5)
        self.experiment_layout.addWidget(self.start_experiment_button, 1, 0)
        self.experiment_layout.addWidget(self.stop_experiment_button, 2, 0)
        self.experiment_layout.addWidget(self.directory_input, 3, 0)
        self.experiment_layout.addWidget(self.browse_button, 3, 1)
        self.experiment_layout.addWidget(voltagelabel, 4, 0)
        self.experiment_layout.addWidget(self.voltagevalues, 4, 1)
        self.experiment_layout.addWidget(self.TurnOnVolts, 5, 0)
        


        self.Timelapse_layout.addLayout(self.experiment_layout)


        # Finalizing GUI
        
        self.setWindowTitle('Microscope Control')
        self.show()
        

                                                # All functions for controlling the microscope go here


    def startstoplive_imaging(self):
        if self.live_Button.isChecked() == True:
            self.live_Button.setStyleSheet("background-color: #2ac555;")
            self.video_thread = VideoThread()
        # connect its signal to the update_image slot
            self.video_thread.change_pixmap_signal.connect(self.update_image)
            # start the thread
            self.video_thread.start()
        else:
            self.video_thread.stop()
            self.live_Button.setStyleSheet("background-color: #bb283a;")

    
    def handle_record_button(self):
        if self.record_button.isChecked():
            self.record_button.setText("Recording")
            self.record_button.setStyleSheet("background-color: #2ac555;")
            self.video_thread.start_recording()
        else:
            self.record_button.setText("Not Recording")
            self.record_button.setStyleSheet("background-color: #bb283a;")
            self.video_thread.stop_recording()

        
    def snap_DIA_image(self):
        
        self.image_path = "C:\\Users\\Cell Culture Scope\\Pictures\\image.png" # Your image path
        # set(self.DIAshutterlight,'Value',1)
        #self.mmc.setConfig('TimeLapseJulia','DIA') # Load config
        self.mmc.setProperty(self.core,'Shutter',self.DIAshutter)
        self.mmc.waitForSystem()
        self.mmc.setExposure(self.camera, 20) # Load Exposure
        self.mmc.setProperty(self.camera,'CONVERSION FACTOR COEFF', 0.5) # Load gain
        self.mmc.setProperty(self.lightpath,'Label',self.camerapath)
        #self.mmc.setProperty(self.DIAlamp, 'State', 1)
        #self.mmc.setProperty(self.DIAlamp, 'Intensity', 4)
        # self.mmc.setProperty(self.camera,'Exposure',self.num_expotimelapse_dia);
        #self.mmc.waitForConfig('TimeLapse','DIA')
        self.mmc.waitForDevice(self.filter) # Kinda custom wait for the slowest device that would allow snapping an image with everything ok
        self.mmc.waitForSystem()
        #self.mmc.waitForImageSynchro()
        self.mmc.setAutoShutter(True)

        self.mmc.snapImage() # Snap Image
        rawImage = self.mmc.getImage() # Process Image
        image_width = self.mmc.getImageWidth()
        image_height = self.mmc.getImageHeight()
        print(type(self.mmc.getBytesPerPixel))
        #if int(self.mmc.getBytesPerPixel) == 2:
        pixelType = np.uint16
        #else:
            #pixelType = np.uint8

        # Assuming rawImage is a byte array
        rawImage = np.frombuffer(rawImage, dtype= pixelType)
        rawImage = rawImage.reshape((image_height, image_width)).T
        # Assuming rawImage is a numpy array
        self.adjusted_image = exposure.rescale_intensity(rawImage)
        self.fake_save_image(self.adjusted_image) #Hard save image before display to compare
        raw_data = self.adjusted_image.tobytes()
        self.myQImage = QImage(raw_data, image_width, image_height, QImage.Format_Grayscale16)
        self.final_Image = QPixmap.fromImage(self.myQImage)
        self.image_Live.setPixmap(self.final_Image)
        #self.mmc.setProperty(self.DIAlamp, 'State', 0) # Lamp takes time to turn ON
    


    def snap_EPI_image(self, Filter, Exposure_value):
        
        self.image_path = "C:\\Users\\Cell Culture Scope\\Pictures\\image.png" # Your image path
        self.mmc.setProperty(self.core,'Shutter',self.EPIshutter)
        self.mmc.waitForSystem()
        self.change_filter(Filter)
        self.mmc.setExposure(self.camera, Exposure_value) # Load Exposure
        #self.mmc.setProperty(self.camera,'Gain', 20) # Load gain
        self.mmc.setProperty(self.lightpath,'Label',self.camerapath)
        self.mmc.waitForDevice(self.filter) # Kinda custom wait for the slowest device that would allow snapping an image with everything ok
        self.mmc.waitForSystem()
        #self.mmc.waitForImageSynchro()
        #self.mmc.setAutoShutter(True)

        self.mmc.snapImage() # Snap Image
        rawImage = self.mmc.getImage() # Process Image
        image_width = self.mmc.getImageWidth()
        image_height = self.mmc.getImageHeight()
        print(type(self.mmc.getBytesPerPixel))
        #if int(self.mmc.getBytesPerPixel) == 2:
        pixelType = np.uint16
        #else:
            #pixelType = np.uint8

        # Assuming rawImage is a byte array
        rawImage = np.frombuffer(rawImage, dtype= pixelType)
        rawImage = rawImage.reshape((image_height, image_width)).T
        # Assuming rawImage is a numpy array
        self.adjusted_image = exposure.rescale_intensity(rawImage)
        self.fake_save_image(rawImage) #Hard save image before display to compare
        raw_data = rawImage.tobytes()
        raw_data_rescaled = self.adjusted_image.tobytes()
        self.myQImage = QImage(raw_data, image_width, image_height, QImage.Format_Grayscale16)
        self.myQImage.save('C:\\Users\\Cell Culture Scope\\Pictures\\image_Qimage.png')
        self.myQImage_rescaled = QImage(raw_data_rescaled, image_width, image_height, QImage.Format_Grayscale16)
        self.final_Image = QPixmap.fromImage(self.myQImage_rescaled)
        self.image_Live.setPixmap(self.final_Image) #Show adjusted image on app but only save image with real values
        return self.myQImage
    


    def save_image(self, save_path):
        if self.image_Live.pixmap():
            self.image_Live.pixmap().save(self.image_path)
            print(f"Image saved to {self.image_path}")
        else:
            print("No image to save")
    

    def fake_save_image(self, fed_image):
        image = Image.fromarray(fed_image, mode='I;16')

        # Save as PNG
        image.save('C:\\Users\\Cell Culture Scope\\Pictures\\image2.png')
         
             
    @pyqtSlot(np.ndarray)
    def update_image(self, cv_img):
        """Updates the image_label with a new opencv image"""
        qt_img = self.convert_np_qt(cv_img)

        self.image_Live.setPixmap(qt_img)

    def convert_np_qt(self, np_img):
        """Convert from an np image to QPixmap"""
        h, w = np_img.shape
        raw_data = np_img.tobytes()
        self.myQImage = QImage(raw_data, w, h, QImage.Format_Grayscale16)
        return QPixmap.fromImage(self.myQImage)

    def DIAlamp_ON(self):
         if self.dialamponlight.isChecked() == True:
              self.mmc.setProperty(self.DIAlamp, 'State', 1)
              self.dialamponlight.setStyleSheet("background-color: #2ac555;")
         else:
              self.mmc.setProperty(self.DIAlamp, 'State', 0)
              self.dialamponlight.setStyleSheet("background-color: #bb283a;")

    def save_Position(self):
        # Save a new position
        if len(self.positions) < 8:
            x, y, z = self.get_new_position()  # Simulated new position
            self.positions.append([x, y, z])
            self.updateTable()
        else:
            QMessageBox.warning(self, 'Limit Reached', 'Cannot load more than 8 positions.')

    def get_new_position(self):
        # Simulate getting a new position (replace this with actual microscope data fetching)
        return self.mmc.getXPosition(self.stage), self.mmc.getYPosition(self.stage), self.mmc.getPosition()
    

    def updateTable(self):
        # Update the table with the current positions
        for row, [x, y, z] in enumerate(self.positions):
            self.Positions_table.setItem(row, 0, QTableWidgetItem(f'{x:.2f}'))
            self.Positions_table.setItem(row, 1, QTableWidgetItem(f'{y:.2f}'))
            self.Positions_table.setItem(row, 2, QTableWidgetItem(f'{z:.2f}'))

        # Clear unused rows
        for row in range(len(self.positions), 8):
            self.Positions_table.setItem(row, 0, QTableWidgetItem(''))
            self.Positions_table.setItem(row, 1, QTableWidgetItem(''))
            self.Positions_table.setItem(row, 2, QTableWidgetItem(''))

    def clearPositions(self):
        # Clear all saved positions
        self.positions = []
        self.updateTable()
    

    def replacePosition(self):
        # Get user input to select the row number to replace
        if not self.positions:
            QMessageBox.warning(self, 'No Positions', 'No positions available to replace.')
            return

        row_number, ok = QInputDialog.getInt(self, 'Replace Position', 'Enter position number (1-8):', 1, 1, len(self.positions))

        if ok:
            row_index = row_number - 1  # Convert to zero-based index
            if row_index < len(self.positions):
                x, y, z = self.get_new_position()  # Simulated new position
                self.positions[row_index] = [x, y, z]
                self.updateTable()
            else:
                QMessageBox.warning(self, 'Invalid Input', 'Position does not exist.')



    
    def PathtoCamera(self):
        self.mmc.setProperty(self.lightpath, 'Label', self.camerapath)
        
    def Set_zoom(self,item):
        self.mmc.setProperty(self.zoom, 'Label', item)
    
    def change_filter(self, filter_name):
        chosen_filter = filter_name
        #chosen_filter = [int(num) for num in re.findall(r'\d+', filter_name)]
        self.mmc.setProperty(self.filter, 'State', chosen_filter - 1)
    
    def Set_stage_speed(self, item):
        self.stage_speed = int(item)
    
    def toggle_DIA_shutter(self):
        if self.DIAshutterbutton.isChecked() == True:
            self.mmc.setProperty(self.DIAshutter, 'State', 1)
            self.DIAshutterbutton.setStyleSheet("background-color: #2ac555;")
        else:
            self.mmc.setProperty(self.DIAshutter, 'State', 0)
            self.DIAshutterbutton.setStyleSheet("background-color: #bb283a;")
    
    def toggle_EPI_shutter(self):
        if self.EPIshutterbutton.isChecked() == True:
            self.mmc.setProperty(self.EPIshutter, 'State', 1)
            self.EPIshutterbutton.setStyleSheet("background-color: #2ac555;")
        else:
            self.mmc.setProperty(self.EPIshutter, 'State', 0)
            self.EPIshutterbutton.setStyleSheet("background-color: #bb283a;")



                                                    # All functions for controlling valves go here

                                                    

    

    def control_valve(self, idx, state):
        """Handles Control button toggle actions."""
        # Relay index mapping for serial communication
        relay_id = self.get_relay_id(idx)

        if state:
            # Turn the valve on
            self.controls[idx].setStyleSheet("background-color: green")
            self.send_relay_command(f"relay off {relay_id}")
        else:
            # Turn the valve off
            self.controls[idx].setStyleSheet("background-color: red")
            self.send_relay_command(f"relay on {relay_id}")

    def get_relay_id(self, idx):
        """Maps relay 10-21 to A-L for serial communication, keeps others unchanged."""
        if idx <= 9:  # For relays 0 to 9
            return str(idx)
        else:  # For relays 10 to 21, map to A-L
            return chr(ord('A') + (idx - 10))

    def stop_all_callback(self):
        """Handles StopALL button press, turning off all valves."""
        self.stop_all_button.setStyleSheet("background-color: green")

        # Turn off all controls
        for i, control in enumerate(self.controls):
            relay_id = self.get_relay_id(i)
            control.setStyleSheet("background-color: red")
            control.setChecked(False)
            self.send_relay_command(f"relay on {relay_id}")

        # Optionally send a command to close all relays at once
        self.send_relay_command('open all')

        # Restore StopALL button to its original color
        self.stop_all_button.setStyleSheet("background-color: red")

    def all_on_callback(self):
        """Handles the 'All On' button press, turning on all valves."""
        self.all_on_button.setStyleSheet("background-color: green")

        # Turn on all controls
        for i, control in enumerate(self.controls):
            relay_id = self.get_relay_id(i)
            control.setStyleSheet("background-color: green")
            control.setChecked(True)
            self.send_relay_command(f"relay off {relay_id}")

        # Optionally send a command to open all relays at once
        self.send_relay_command('close all')

    def send_relay_command(self, command):
        """Sends a command to the serial device."""
        if self.Numato_port and self.Numato_port.is_open:
            try:
                self.Numato_port.write(f"{command}\r".encode('utf-8'))
                #self.Numato_port.flush()
                time.sleep(0.005)
            except serial.SerialException as e:
                QMessageBox.critical(self, 'Error', 'Failed to communicate with the device')

    def Characterize_Droplet(self, input, chemostat_number=1, purge_duration=0, flow_duration=0, drive_duration=0):

        drive_duration = float(self.drive_duration_Input.text())
        purge_duration = float(self.purge_duration_Input.text())
        flow_duration = float(self.flow_duration_Input.text())
        chemostat_number = int(self.inlet_Input.text())

        self.Characterize_Droplet_thread = DropletWorker("characterize", input, 
                                                         purge_duration = purge_duration, 
                                                         flow_duration = flow_duration, 
                                                         drive_duration = drive_duration,
                                                         chemostat_number = chemostat_number)
        self.Characterize_Droplet_thread.start()


    def PWM_droplet(self, input_1 = 12, input_2 = 13, PWM_duration1 = 0.05, PWM_duration2 = 0.05, PWM_totalduration = 5.0):

        PWM_duration1 = float(self.PWM_duration1_Input.text())
        PWM_duration2 = float(self.PWM_duration2_Input.text())
        PWM_totalduration = float(self.PWM_totalduration_Input.text())

        self.PWM_thread = DropletWorker("PWM", input_1= input_1, input_2 = input_2, 
                                        PWM_duration1= PWM_duration1, PWM_duration2= PWM_duration2, 
                                        PWM_totalduration= PWM_totalduration)
        self.PWM_thread.start()

    def alter_Arduino_state(self, checked):
        if checked == True:
            self.TurnOnVolts.setText('Volts ON')
            self.TurnOnVolts.setStyleSheet("background-color: #2ac555;")
            send_to_arduino(2)
        else:
            self.TurnOnVolts.setText('Volts OFF')
            self.TurnOnVolts.setStyleSheet("background-color: #bb283a;")
            send_to_arduino(3)


                                               # All functions necessary for timelapse imaging go here


    def read_Exposure_values(self):
            """
            Read values from the Exposure table's cells with spinboxes.
            """
            table = self.Exposures_table
            rows = table.rowCount()
            cols = table.columnCount()

            self.selected_exposures = []
            for row in range(self.Exposures_table.rowCount()):
                filter_item = table.cellWidget(row, 0).value()  # Filter name (number)
                exposure_item = table.cellWidget(row, 1).value()  # Exposure time

                if filter_item and exposure_item:  # Ensure both items exist
                    if exposure_item > 0:  # Check if the exposure time is non-zero
                            self.selected_exposures.append([filter_item, exposure_item])  # Store [Filter Number, Exposure Time]


            print("Table Values:")
            self.interval = int(self.cycle_Interval_Input.text())
            self.cycles = int(self.cycle_Input.text())
            print(self.selected_exposures, self.interval, self.cycles)

    def create_centered_checkbox(self):
        """
        Creates a centered checkbox inside a QFrame to ensure it is aligned properly in the cell.
        """
        frame = QFrame()
        layout = QHBoxLayout(frame)
        layout.setContentsMargins(0, 0, 0, 0)  # Remove padding
        layout.setAlignment(Qt.AlignCenter)  # Center-align the layout

        checkbox = QCheckBox(frame)
        checkbox.setEnabled(False)  # Disable checkbox for display only
        layout.addWidget(checkbox)

        return frame
    

    def add_loading_step(self):
        # Collect selected inputs and rings
        selected_inputs = [spinbox.value() for spinbox in self.Chemostat_inputs if spinbox.value() > 0]
        ring_status = [checkbox.isChecked() for checkbox in self.Chemostats]
        
        if not selected_inputs and not any(ring_status):
            return  # Ignore if no inputs or rings are selected
        
        # Store the step
        step = {
            "input1": selected_inputs[0] if len(selected_inputs) > 0 else "",
            "input2": selected_inputs[1] if len(selected_inputs) > 1 else "",
            "rings": ring_status
        }
        self.Chemostat_protocol_steps.append(step)
        
        # Save a step in the next available column
        col_index = self.current_protocol_table_step
        #self.Chemostat_protocol_table.insertColumn(col_index)
        
        
        # Populate the column with step data
        input_1 = QTableWidgetItem(str(step["input1"]))
        input_1.setTextAlignment(Qt.AlignCenter)
        input_2 = QTableWidgetItem(str(step["input2"]))
        input_2.setTextAlignment(Qt.AlignCenter)
        self.Chemostat_protocol_table.setItem(0, col_index, input_1)
        self.Chemostat_protocol_table.setItem(1, col_index, input_2)
        
        for i, is_on in enumerate(step["rings"]):
            checkbox = QCheckBox()
            checkbox.setChecked(is_on)
            checkbox.setEnabled(False)  # Disable interaction
            widget = QWidget()
            layout = QVBoxLayout(widget)
            layout.addWidget(checkbox)
        
            # Center align the checkbox in the cell
            layout.setContentsMargins(0, 0, 0, 0)
            layout.setAlignment(checkbox, Qt.AlignCenter)
            self.Chemostat_protocol_table.setCellWidget(2 + i, col_index, widget)

        self.Chemostat_protocol_table.resizeColumnsToContents 
        self.AdjustTableSize(self.Chemostat_protocol_table)
        self.current_protocol_table_step += 1
    
    def clear_all_steps(self):
        # Clear the steps and reset the table
        self.Chemostat_protocol_steps = []
        self.Chemostat_protocol_table.setColumnCount(0)
    
    def clear_last_step(self):
        # Remove the last step from the data and table
        if self.Chemostat_protocol_steps:
            self.Chemostat_protocol_steps.pop()
            if self.current_protocol_table_step > 0:
                self.current_protocol_table_step -= 1
                col = self.current_protocol_table_step

                for row in range(10):
                    if row < 2:  # Clear text in first 2 rows
                        self.Chemostat_protocol_table.setItem(row, col, QTableWidgetItem(""))
                    else:  # Clear checkboxes
                        frame = self.Chemostat_protocol_table.cellWidget(row, col)
                        checkbox = frame.layout().itemAt(0).widget()  # Access the checkbox in the layout
                        if checkbox:
                            checkbox.setChecked(False)
    
    def export_to_csv(self):
        if not self.Chemostat_protocol_steps:
            return  # Ignore if no steps to export
        
        # Open a file dialog to select save location
        options = QFileDialog.Options()
        file_path, _ = QFileDialog.getSaveFileName(
            self, "Save Sequence to CSV", "", "CSV Files (*.csv);;All Files (*)", options=options
        )
        
        if not file_path:
            return  # User canceled saving
        
        # Write data to CSV
        with open(file_path, mode="w", newline="") as file:
            writer = csv.writer(file)
            
            # Write headers
            headers = ["Step", "Input 1", "Input 2"] + [f"Ring {i + 1}" for i in range(8)]
            writer.writerow(headers)
            
            # Write step data
            for step_index, step in enumerate(self.Chemostat_protocol_steps, start=1):
                row = [
                    f"Step {step_index}",
                    step["input1"],
                    step["input2"]
                ] + ["ON" if status else "OFF" for status in step["rings"]]
                writer.writerow(row)


    def resize_table_to_fit_contents(self, table):
        # Get the total width of all columns
        total_width = sum(table.columnWidth(i) for i in range(table.columnCount()))
        # Add the vertical scrollbar width if present
        if table.verticalScrollBar().isVisible():
            total_width += table.verticalScrollBar().width()

        # Get the total height of all rows
        total_height = sum(table.rowHeight(i) for i in range(table.rowCount()))
        # Add the horizontal header height
        total_height += table.horizontalHeader().height()
        # Add the horizontal scrollbar height if present
        if table.horizontalScrollBar().isVisible():
            total_height += table.horizontalScrollBar().height()

        # Resize the table to fit its contents
        table.setFixedSize(total_width, total_height)
    
    def AdjustTableSize(self, table):
        w = table.verticalHeader().width() + 4  # +4 seems to be needed
        for i in range(table.columnCount()):
            w += table.columnWidth(i)  # seems to include gridline (on my machine)
        h = table.horizontalHeader().height() + 4
        for i in range(table.rowCount()):
            h += table.rowHeight(i)
        table.setMaximumSize(w, h)
        table.setMinimumSize(w, h)

    def browse_folder(self):
        folder = QFileDialog.getExistingDirectory(self, "Select Folder")
        if folder:  # If a folder was selected
            os.chdir(folder)
            self.directory_input.setText(folder)


    def TimeLapse_Experiment(self, num_loops, time_interval, positions_table, selected_exposures, chemostat_protocol_table):
        """
        Performs imaging and executes a chemostat protocol in a loop.
        
        Args:
            num_loops (int): Number of loops to perform.
            time_interval (float): Time interval between each loop in seconds.
            positions_table (list): A list containing the positions to image.
            chemostat_protocol_table (list): A list of steps for the chemostat protocol.

        Example:
            positions_table = [{"Position": "Pos1", "X": 100, "Y": 200}, {"Position": "Pos2", "X": 150, "Y": 250}]
            chemostat_protocol_table = [
                {"Step": 1, "Action": "Open Valve", "Target": "Valve1"},
                {"Step": 2, "Action": "Close Valve", "Target": "Valve1"}
            ]
        """
        # Initialize all valves

        self.control_valve(0, state = False)
        self.control_valve(1, state = True)
        self.control_valve(2, state = False)
        self.control_valve(3, state = True)
        self.control_valve(4, state = True)
        self.control_valve(5, state = False)
        self.control_valve(6, state = False)
        self.control_valve(7, state = True)
        self.control_valve(8, state = False)
        self.control_valve(9, state = False)
        self.control_valve(10, state = False)
        self.control_valve(11, state = False)
        self.control_valve(12, state = True)
        self.control_valve(13, state = True)
        self.control_valve(14, state = True)
        self.control_valve(15, state = True)

        purge_duration = float(self.purge_duration_Input.text())
        flow_duration = float(self.flow_duration_Input.text())
        drive_duration = float(self.drive_duration_Input.text())
        
        for loop in range(num_loops):
            print(f"--- Starting loop {loop + 1} of {num_loops} ---")

            loop_start_time = time.time()
            
            
            # Step 1: Iterate through positions and image each position
            for cur_index in range(len(positions_table)):
                self.mmc.setXYPosition(positions_table[cur_index][0], positions_table[cur_index][1])
                self.mmc.setPosition(positions_table[cur_index][2])
                self.mmc.waitForSystem()
                time.sleep(0.5)
                for i in range(len(selected_exposures)):
                    Image = self.snap_EPI_image(Filter = selected_exposures[i][0], Exposure_value= selected_exposures[i][1])
                    file_name = f"Expt_{cur_index + 1}_{selected_exposures[i][0]}_{selected_exposures[i][1]}_{loop + 1}.tiff"
                    save_path = os.path.join(".", file_name)
                    time.sleep(0.2)
                    if Image.save(save_path):
                        print(f"Image saved successfully as: {save_path}")
                    else:
                        print("Failed to save the image.")

            
            
            # Step 2: Execute the protocol steps
            
            
            if self.Chemostat_protocol_steps != []:
                print("Executing chemostat protocol:")
                self.control_valve(15, state = False)
                time.sleep(30)
                self.control_valve(15, state = True)
                
                self.control_valve(7, state = False)
                               

                self.control_valve(12, state = False)
                time.sleep(5)
                self.control_valve(12, state = True)

                self.control_valve(13, state = False)
                time.sleep(5)
                self.control_valve(13, state = True)
                
                self.control_valve(14, state = False)
                time.sleep(5)
                self.control_valve(14, state = True)

                self.control_valve(15, state = False)
                time.sleep(5)
                self.control_valve(15, state = True)

                self.control_valve(7, state = True)

                self.mmc.setProperty(self.DIAlamp, 'State', 1) #Turn on DIA lamp for recording droplet merging events
                self.video_thread = VideoThread()
                self.video_thread.start()
                for step in range(len(self.Chemostat_protocol_steps)):
                    input_1 = self.Chemostat_protocol_steps[step]["input1"]
                    input_2 = self.Chemostat_protocol_steps[step]["input2"]
                    Purge_worker = DropletWorker("purge", input_1, input_2= input_2, purge_duration = purge_duration)
                    Purge_worker.start()
                    Purge_worker.wait()
                    #input_2 = self.Chemostat_protocol_steps[step]["input2"]
                    rings_chosen = self.Chemostat_protocol_steps[step]["rings"]
                    for ring_number in range(len(rings_chosen)):
                        if rings_chosen[ring_number]:
                            self.mmc.setXYPosition(positions_table[ring_number][0], positions_table[ring_number][1]) 
                            self.mmc.setPosition(positions_table[ring_number][2]) #Go to chemostat #x position
                            Generate_worker = DropletWorker("generate", input_1, flow_duration = flow_duration)
                            Generate_worker.start()
                            Generate_worker.wait()
                            self.video_thread.start_recording()
                            
                            Drive_worker = DropletWorker("drive", input_1, drive_duration = drive_duration, chemostat_number = ring_number + 1)
                            Drive_worker.start()
                            Drive_worker.wait()
                            
                            self.video_thread.stop_recording()
                            
                            print(f"Generated droplet from {input_1} and fed to Chemostat {ring_number + 1}")
                    self.control_valve(15, state = False)
                    self.control_valve(7, state = False)
                    time.sleep(5)
                    self.control_valve(15, state = True)
                    self.control_valve(7, state = True)
                    
                self.video_thread.stop()
                self.mmc.setProperty(self.DIAlamp, 'State', 0)
                Purge_worker = None
                Generate_worker = None
                Drive_worker = None
                Wash_worker = None
                            
             # Calculate the time spent on steps 1 and 2
            loop_elapsed_time = time.time() - loop_start_time
            #self.control_valve(6, state = True)


            # Step 3: Wait for the remaining time before the next loop
            remaining_time = time_interval * 60 - loop_elapsed_time  # Subtract elapsed time from the total interval
            if int(remaining_time) > 0:
                print(f"Waiting for {remaining_time:.2f} seconds before the next loop.")
                time.sleep(remaining_time)  # Wait for the remaining time

        print("--- Experiment complete ---")
        
        self.control_valve(0, state = True)



    def closeEvent(self, event):
        """Overrides close event to ensure the serial connection is closed."""
        self.all_on_callback()
        if self.Numato_port and self.Numato_port.is_open:
            self.Numato_port.close()
        #self.mmc.setProperty('IntensiLightShutter', 'State', 0)
        arduino.close()
        event.accept()
    

    



if __name__ == '__main__':
    import sys
    app = QtWidgets.QApplication(sys.argv)
    app.setStyle('Fusion')
    window = MicroscopeControlGUI()
    window.show()
    sys.exit(app.exec_())
